# Notebook 2: European Options Pricing
**Project:** Monte Carlo Risk & Derivatives Pricing
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations')


Loaded 2,609 observations


In [2]:
from options_pricing import bs_call, bs_put, mc_european
S0,K,r,T,sigma = 100,100,0.05,1.0,0.20
bs_c = bs_call(S0,K,T,r,sigma)
mc_c, mc_se = mc_european(S0,K,T,r,sigma,n_sims=100000)
print(f'Black-Scholes Call : ${bs_c:.4f}')
print(f'Monte Carlo Call   : ${mc_c:.4f} (SE={mc_se:.4f})')
print(f'Pricing Error      : ${abs(bs_c-mc_c):.4f}')

Black-Scholes Call : $10.4506
Monte Carlo Call   : $10.4739 (SE=0.0490)
Pricing Error      : $0.0233


In [3]:
strikes=np.arange(80,125,5)
results=[]
for K in strikes:
    bs_c=bs_call(S0,K,T,r,sigma)
    mc_c,mc_se=mc_european(S0,K,T,r,sigma,n_sims=100000)
    results.append({'Strike':K,'BS Call':round(bs_c,4),'MC Call':round(mc_c,4),'SE':round(mc_se,4)})
opts_df=pd.DataFrame(results).set_index('Strike')
opts_df.to_csv('../results/options_pricing_results.csv')
print(opts_df.to_string())

        BS Call  MC Call      SE
Strike                          
80      24.5888  24.6105  0.0638
85      20.4693  20.4915  0.0612
90      16.6994  16.7228  0.0579
95      13.3465  13.3702  0.0537
100     10.4506  10.4739  0.0490
105      8.0214   8.0416  0.0439
110      6.0401   6.0603  0.0387
115      4.4666   4.4863  0.0336
120      3.2475   3.2650  0.0288


In [4]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
ax1.plot(opts_df.index,opts_df['BS Call'],'o-',color='#185FA5',linewidth=2,markersize=7,label='Black-Scholes')
ax1.plot(opts_df.index,opts_df['MC Call'],'s--',color='#E24B4A',linewidth=2,markersize=7,label='Monte Carlo')
ax1.fill_between(opts_df.index,opts_df['MC Call']-2*opts_df['SE'],opts_df['MC Call']+2*opts_df['SE'],alpha=0.2,color='#E24B4A',label='MC 95% CI')
ax1.axvline(S0,color='gray',linewidth=1,linestyle='--',alpha=0.6,label='ATM')
ax1.set_title('European Call: Black-Scholes vs Monte Carlo'); ax1.legend(fontsize=8)
ax2=axes[1]
sigmas=[0.15,0.20,0.25,0.30]
for sig,c in zip(sigmas,["#185FA5","#E24B4A","#1D9E75","#7B1FA2"]):
    calls=[bs_call(S0,K,T,r,sig) for K in np.arange(80,125,5)]
    ax2.plot(np.arange(80,125,5),calls,'o-',color=c,linewidth=1.8,markersize=5,label=f'sigma={int(sig*100)}%')
ax2.set_title('Call Price vs Strike — Multiple Volatilities'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/03_european_option_pricing.png',dpi=150,bbox_inches='tight')
plt.show()